# Lab 2. Notebook 03 — Train/Test Split and Preprocessing Pipelines

**Course:** Artificial Intelligence with Python  
**Estimated time:** 55 minutes  
**Prerequisites:** Lab 2 Notebooks 01 and 02  
**Dataset:** `data/apartments_corrupted.csv`

## Learning objectives

By the end of this notebook you should be able to:

1. Separate **features (`X`)** from a **target (`y`)** for a supervised learning problem.
2. Create reproducible **training** and **test** sets.
3. Explain why fitting preprocessing operations before the split causes **data leakage**.
4. Build a `Pipeline` for numerical and categorical features, and combine them using `ColumnTransformer`.
5. Apply preprocessing consistently to training and test observations without refitting on test data.

**Today's goal:** Prepare the dataset to predict monthly apartment rent. You will **not train a prediction model yet** — that comes in Lab 3.

> **Important:** Work through the notebook in order. Complete the exercises in their code cells. The guided examples run independently of your TODO solutions.

## 1. Load the data and apply rule-based cleaning (10 min)

In Notebook 01 you found missing entries, duplicates, inconsistent categories, and invalid numbers. In Notebook 02 you learned imputation, encoding, and scaling on **toy data**.

This time we will work with the actual apartment CSV. We first apply **deterministic, row-by-row rules** (which do not estimate averages, medians, ranges, or category frequencies). Then we split the data **before** fitting any statistical preprocessing.

Our simplified classroom rules are explicit, rather than a universal cleaning recipe:

- Remove identical repeated rows.
- If the same apartment ID describes conflicting rows, exclude those ambiguous records for manual review.
- Convert malformed numeric text to missing values (`NaN`), and replace impossible measurements with `NaN`.
- Normalize known text categories.
- Omit rows without a valid positive target rent; **never impute the target**.
- Keep unusually high *positive* observations unless we have evidence that they are errors.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

csv_path = Path("data/apartments_corrupted.csv")
if not csv_path.is_file():
    raise FileNotFoundError(
        "Open this notebook from the lab02 directory so data/apartments_corrupted.csv is available."
    )

raw = pd.read_csv(csv_path)
print("Original shape:", raw.shape)
display(raw.head())

Original shape: (502, 9)


,apartment_id,area_m2,rooms,building_age_years,distance_to_center_km,district,balcony,energy_rating,monthly_rent_eur
0,APT-0001,123.2,4,19,5.3,West,yes,A,2134.0
1,APT-0002,58.2,1,67,13.7,East,no,D,845.0
2,APT-0003,98.2,3,7,14.4,West,no,D,1920.0
3,APT-0004,89.4,4,4,13.5,East,no,D,1668.0
4,APT-0005,53.2,2,60,8.4,South,no,D,868.0


In [2]:
# Deterministic cleaning: no mean/median/scaling learned from the dataset.
df = raw.drop_duplicates().copy()

# Conflicting records for the same ID are ambiguous; exclude both for this exercise.
conflicting_ids = df.loc[df["apartment_id"].duplicated(keep=False), "apartment_id"].unique()
print("IDs requiring manual review:", list(conflicting_ids))
df = df.loc[~df["apartment_id"].isin(conflicting_ids)].copy()

numeric_columns = [
    "area_m2", "rooms", "building_age_years",
    "distance_to_center_km", "monthly_rent_eur"
]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Reject physically impossible feature values, but leave missing values for imputation.
for col, invalid in {
    "area_m2": df["area_m2"] <= 0,
    "rooms": df["rooms"] < 1,
    "building_age_years": df["building_age_years"] < 0,
    "distance_to_center_km": df["distance_to_center_km"] < 0,
}.items():
    df.loc[invalid, col] = np.nan

# Normalize the text without learning population statistics.
df["district"] = df["district"].astype("string").str.strip().str.title()
df["energy_rating"] = df["energy_rating"].astype("string").str.strip().str.upper()

balcony_text = df["balcony"].astype("string").str.strip().str.lower()
df["balcony"] = balcony_text.map({
    "yes": "yes", "y": "yes", "true": "yes",
    "no": "no", "n": "no", "false": "no"
})

# scikit-learn's categorical SimpleImputer expects np.nan rather than pd.NA.
for col in ["district", "balcony", "energy_rating"]:
    df[col] = df[col].astype(object).where(df[col].notna(), np.nan)

# Rent is the answer we want the model to predict. A missing/negative rent
# cannot provide a useful training label in this exercise.
df = df.loc[df["monthly_rent_eur"].notna() & (df["monthly_rent_eur"] > 0)].copy()

print("Usable rows:", len(df))
print("Missing feature values after rule-based cleaning:")
display(df.isna().sum())

IDs requiring manual review: ['APT-0455']
Usable rows: 494
Missing feature values after rule-based cleaning:


apartment_id             0
area_m2                  8
rooms                    2
building_age_years       5
distance_to_center_km    3
district                 1
balcony                  2
energy_rating            2
monthly_rent_eur         0
dtype: int64

### Exercise 1 — Inspect the remaining issues (3 min)

1. Check whether any duplicate apartment IDs remain.
2. Count the remaining missing values in `area_m2`, `district`, and `balcony`.
3. Explain why replacing missing `area_m2` values with the **overall dataset median at this point** would be a mistake.

**Hint:** A statistical imputer must learn its median only from the training data.

In [3]:
# TODO: count duplicate IDs after cleaning
print("Duplicate IDs:", df["apartment_id"].duplicated().sum())

# TODO: count missing area, district and balcony values
print(df[["area_m2", "district", "balcony"]].isna().sum())

# TODO: write a one-sentence leakage explanation in a Markdown cell

Duplicate IDs: 0
area_m2     8
district    1
balcony     2
dtype: int64


**Your explanation:** Computing the median on the full dataset before splitting would use information from the future test rows, so the test set would no longer be truly unseen (data leakage) and the evaluation would be over-optimistic.

## 2. Define features and target; split the dataset (10 min)

To predict rent, we need to separate:

- `X`: the input attributes (**features**), such as size, rooms, district, and balcony.
- `y`: the known output (**target**), `monthly_rent_eur`.

We deliberately exclude `apartment_id`: an arbitrary identifier is not a meaningful apartment characteristic and could tempt a model to memorize particular records.

We reserve **20%** of the observations as an unseen **test set**. Use a fixed `random_state` to reproduce the same split on different computers.

In [4]:
from sklearn.model_selection import train_test_split

target_col = "monthly_rent_eur"
id_col = "apartment_id"

X = df.drop(columns=[target_col, id_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Full feature matrix:", X.shape)
print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
print("Training targets:", y_train.shape)
print("Test targets:", y_test.shape)
assert X_train.index.intersection(X_test.index).empty
assert X_train.shape[1] == X_test.shape[1]

Full feature matrix: (494, 7)
Training features: (395, 7)
Test features: (99, 7)
Training targets: (395,)
Test targets: (99,)


### Exercise 2 — Explore a different split (4 min)

1. Calculate the percentage of rows assigned to training and testing.
2. Repeat the split with `test_size=0.25` and `random_state=42`, using variables ending in `_25`.
3. What happens if you change `random_state`? Does the **number** of test rows change?

Do not overwrite `X_train` or `X_test`; the next sections will use the original 80/20 split.

In [5]:
# TODO: print training and test percentages for the original split
n_total = len(X)
print(f"Train: {len(X_train) / n_total:.1%} | Test: {len(X_test) / n_total:.1%}")

# TODO: create X_train_25, X_test_25, y_train_25, y_test_25
X_train_25, X_test_25, y_train_25, y_test_25 = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# TODO: compare the row counts and discuss random_state
print("80/20 split -> train:", len(X_train), "test:", len(X_test))
print("75/25 split -> train:", len(X_train_25), "test:", len(X_test_25))

X_train_7, X_test_7, _, _ = train_test_split(X, y, test_size=0.20, random_state=7)
print("random_state=7 -> test rows:", len(X_test_7))
print("Same test rows as random_state=42:", set(X_test_7.index) == set(X_test.index))
print("Changing random_state changes WHICH rows go to the test set, not HOW MANY: the size only depends on test_size.")

Train: 80.0% | Test: 20.0%
80/20 split -> train: 395 test: 99
75/25 split -> train: 370 test: 124
random_state=7 -> test rows: 99
Same test rows as random_state=42: False
Changing random_state changes WHICH rows go to the test set, not HOW MANY: the size only depends on test_size.


## 3. Understand data leakage (8 min)

Imagine predicting the rent of a *future apartment*. You will not know that apartment's effect on the training median or feature scale in advance.

A preprocessing step that calls `fit()` **learns from the data**. For example, `SimpleImputer(strategy="median")` learns a median; `StandardScaler` learns a mean and a standard deviation.

**Incorrect:** split after calculating statistics on the entire dataset.  
**Correct:** split first; learn statistics from `X_train`; transform `X_test` with those **same learned statistics**.

The tiny example below makes the difference visible without modifying our apartment split.

In [6]:
from sklearn.impute import SimpleImputer

# The test set has an unusually large value: 10,000.
toy_train = pd.DataFrame({"area_m2": [50.0, 60.0, np.nan]})
toy_test = pd.DataFrame({"area_m2": [10_000.0, np.nan]})

safe_imputer = SimpleImputer(strategy="median")
safe_imputer.fit(toy_train)       # learn from training data ONLY

wrong_imputer = SimpleImputer(strategy="median")
wrong_imputer.fit(pd.concat([toy_train, toy_test], ignore_index=True))

print("Median learned from training only:", safe_imputer.statistics_[0])
print("Median learned from train + test:", wrong_imputer.statistics_[0])
print("Test transformed safely:", safe_imputer.transform(toy_test).ravel())

Median learned from training only: 55.0
Median learned from train + test: 60.0
Test transformed safely: [10000.    55.]


### Exercise 3 — Explain what `fit()` learns (3 min)

Using the toy example above, answer:

1. Which median would be used to fill a missing test area when training data is used correctly?
2. Why is the median learned from `train + test` invalid for an honest evaluation?
3. Would it be correct to call `fit_transform(toy_test)` with a *new* imputer? Why not?

**Your answers:**

1. The median learned from the training data only: **55.0** (median of 50 and 60).
2. It uses the test value 10,000, so information from the test set leaks into preprocessing; the evaluation would no longer reflect performance on truly unseen data.
3. No. A new imputer fitted on `toy_test` would learn statistics from the test data itself (leakage) and would apply a different transformation than the one used for training, so train and test would be inconsistent.

## 4. Build preprocessing pipelines (17 min)

Previously you used `SimpleImputer`, `OneHotEncoder`, and `StandardScaler` separately. A `Pipeline` connects transformations in the order they should run.

For our apartment data:

- **Numerical:** impute with the training median, then standardize.
- **Categorical:** impute missing values using the training-set most frequent category, then one-hot encode; unseen categories should not crash the pipeline.
- **Combine:** `ColumnTransformer` sends the appropriate columns through their pipelines and joins the results.

We use `sparse_output=False` for the small classroom dataset so that we can inspect the numerical matrix easily. Large datasets may benefit from sparse matrices.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "area_m2", "rooms", "building_age_years", "distance_to_center_km"
]
categorical_features = ["district", "balcony", "energy_rating"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

print("Input numeric features:", numeric_features)
print("Input categorical features:", categorical_features)
print(preprocessor)

Input numeric features: ['area_m2', 'rooms', 'building_age_years', 'distance_to_center_km']
Input categorical features: ['district', 'balcony', 'energy_rating']
ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['area_m2', 'rooms', 'building_age_years',
                                  'distance_to_center_km']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=Fa

We now fit the whole preprocessing system on **training features only**, then apply it to the held-out test features. There is no target (`y`) needed for an unsupervised preprocessing transformation.

In [8]:
X_train_ready = preprocessor.fit_transform(X_train)  # learn + apply
X_test_ready = preprocessor.transform(X_test)       # apply only

feature_names = preprocessor.get_feature_names_out()

X_train_prepared = pd.DataFrame(
    X_train_ready, columns=feature_names, index=X_train.index
)
X_test_prepared = pd.DataFrame(
    X_test_ready, columns=feature_names, index=X_test.index
)

print("Input training shape:", X_train.shape)
print("Output training shape:", X_train_prepared.shape)
print("Output test shape:", X_test_prepared.shape)
display(X_train_prepared.head())

assert X_train_prepared.columns.tolist() == X_test_prepared.columns.tolist()
assert X_train_prepared.notna().all().all()
assert X_test_prepared.notna().all().all()
assert np.isfinite(X_train_ready).all()
assert np.isfinite(X_test_ready).all()

Input training shape: (395, 7)
Output training shape: (395, 16)
Output test shape: (99, 16)


,numeric__area_m2,numeric__rooms,numeric__building_age_years,numeric__distance_to_center_km,categorical__district_Central,categorical__district_East,categorical__district_North,categorical__district_South,categorical__district_West,categorical__balcony_no,categorical__balcony_yes,categorical__energy_rating_A,categorical__energy_rating_B,categorical__energy_rating_C,categorical__energy_rating_D,categorical__energy_rating_E
441,0.610925,1.494646,-0.722767,-0.849851,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
56,-0.848768,-1.604489,-0.607865,0.649898,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
354,1.075994,0.719863,-0.646165,-0.534114,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
224,-0.995323,-1.604489,-0.799368,0.349948,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
365,0.128269,0.719863,1.077363,-1.149800,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


### Examine what the fitted pipeline learned

The numeric imputer stores one median per numeric input column; the numeric scaler stores training means and standard deviations (after imputation). The encoder learns categories present in the training rows.

In [9]:
fitted_numeric = preprocessor.named_transformers_["numeric"]
fitted_categories = preprocessor.named_transformers_["categorical"]

print("Training-set medians:")
print(dict(zip(numeric_features,
               fitted_numeric.named_steps["imputer"].statistics_)))
print("\nDiscovered categories:")
for col, categories in zip(
    categorical_features,
    fitted_categories.named_steps["encoder"].categories_
):
    print(col, "->", list(categories))

print("\nOutput features:", len(feature_names))

Training-set medians:
{'area_m2': np.float64(88.25), 'rooms': np.float64(3.0), 'building_age_years': np.float64(48.0), 'distance_to_center_km': np.float64(9.0)}

Discovered categories:
district -> ['Central', 'East', 'North', 'South', 'West']
balcony -> ['no', 'yes']
energy_rating -> ['A', 'B', 'C', 'D', 'E']

Output features: 16


### Exercise 4 — Inspect the prepared features (5 min)

1. List the names of the encoded `district` columns.
2. Check that **both** prepared DataFrames contain no missing values.
3. Print the mean of each standardized *training* numeric column. They should be approximately zero. Why might the corresponding means in the test data differ from zero?
4. Explain why the output has more columns than the original `X`.

In [10]:
# TODO: list one-hot encoded district column names
district_cols = [c for c in feature_names if c.startswith("categorical__district_")]
print("District columns:", district_cols)

# TODO: verify there are no missing values in X_train_prepared and X_test_prepared
print("Missing in train:", X_train_prepared.isna().sum().sum())
print("Missing in test:", X_test_prepared.isna().sum().sum())

# TODO: print means for numeric columns in training data
numeric_cols = [c for c in feature_names if c.startswith("numeric__")]
print("\nTraining means:")
print(X_train_prepared[numeric_cols].mean().round(6))
print("\nTest means:")
print(X_test_prepared[numeric_cols].mean().round(3))
print("Test means are not exactly 0 because the scaler uses the training mean and std, not the test ones.")

# TODO: explain why the number of columns changed
print("\nOriginal columns:", X.shape[1], "-> prepared columns:", len(feature_names))
print("One-hot encoding replaces each categorical column with one binary column per category, so the number of columns increases.")

District columns: ['categorical__district_Central', 'categorical__district_East', 'categorical__district_North', 'categorical__district_South', 'categorical__district_West']
Missing in train: 0
Missing in test: 0

Training means:
numeric__area_m2                 -0.0
numeric__rooms                    0.0
numeric__building_age_years       0.0
numeric__distance_to_center_km   -0.0
dtype: float64

Test means:
numeric__area_m2                 -0.099
numeric__rooms                   -0.164
numeric__building_age_years       0.013
numeric__distance_to_center_km   -0.160
dtype: float64
Test means are not exactly 0 because the scaler uses the training mean and std, not the test ones.

Original columns: 7 -> prepared columns: 16
One-hot encoding replaces each categorical column with one binary column per category, so the number of columns increases.


## 5. Mini-challenge: Prepare for Lab 3 (10 min)

A new apartment is entered with an unknown district and some missing measurements. Use the **existing, already fitted** `preprocessor` to prepare it for a future prediction. You must not call `.fit()` on this new row.

1. Create the new example (provided below).
2. Transform it with `preprocessor.transform(...)`.
3. Display its numerical feature vector using `feature_names`.
4. Explain what happens to the unseen district (`"Harbor"`) and the missing area.

Then propose a simple **regression baseline** for Lab 3. For example, predict the same training-set median rent for every apartment. Do not train a model today.

In [11]:
new_apartment = pd.DataFrame([{
    "area_m2": np.nan,
    "rooms": 2,
    "building_age_years": 12.0,
    "distance_to_center_km": 3.5,
    "district": "Harbor",  # unseen category
    "balcony": "yes",
    "energy_rating": "B",
}])

# TODO: transform new_apartment with the already fitted preprocessor
new_ready = preprocessor.transform(new_apartment)

# TODO: wrap the result in a DataFrame with feature_names
new_prepared = pd.DataFrame(new_ready, columns=feature_names)
display(new_prepared)

# TODO: inspect the area value and the unknown district encoding
area_median = fitted_numeric.named_steps["imputer"].statistics_[0]
area_scaled = (area_median - fitted_numeric.named_steps["scaler"].mean_[0]) / fitted_numeric.named_steps["scaler"].scale_[0]
print("Scaled area:", new_prepared["numeric__area_m2"].iloc[0], "(training median", area_median, "->", area_scaled, ")")
print("District columns:")
display(new_prepared[district_cols])
print("Missing area -> replaced by the training median, then scaled.")
print("Unseen district 'Harbor' -> all district columns are 0 (handle_unknown='ignore').")

,numeric__area_m2,numeric__rooms,numeric__building_age_years,numeric__distance_to_center_km,categorical__district_Central,categorical__district_East,categorical__district_North,categorical__district_South,categorical__district_West,categorical__balcony_no,categorical__balcony_yes,categorical__energy_rating_A,categorical__energy_rating_B,categorical__energy_rating_C,categorical__energy_rating_D,categorical__energy_rating_E
0,-0.052483,-0.829705,-1.373877,-0.865637,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


Scaled area: -0.05248294921380459 (training median 88.25 -> -0.05248294921380459 )
District columns:


,categorical__district_Central,categorical__district_East,categorical__district_North,categorical__district_South,categorical__district_West
0,0.0,0.0,0.0,0.0,0.0


Missing area -> replaced by the training median, then scaled.
Unseen district 'Harbor' -> all district columns are 0 (handle_unknown='ignore').


**Mini-challenge reflection (write 2–4 sentences):**

- What statistics were learned from the training set?
- Why is it important *not* to refit preprocessing on the new apartment?
- How could you evaluate a regression model in Lab 3 without letting it learn from the held-out test set?

The preprocessor learned the training medians, means and standard deviations of the numeric columns, plus the most frequent value and the list of categories of each categorical column. Refitting on a single new apartment would produce meaningless statistics and a transformation different from the one the model was trained with, so features would no longer be comparable. In Lab 3, a simple baseline is to predict the training median rent for every apartment; any model should beat it. Model choices should be made using only the training set (e.g. with cross-validation), and the test set used once at the end for the final evaluation.

## Wrap-up · Hand-off to Lab 3

You have completed the preprocessing part of the supervised-learning workflow:

**Raw data → rule-based cleaning → define `X` / `y` → train/test split → fit preprocessing on training data → transform test data.**

Check your understanding:

- [ ] I can distinguish an identifier, an input feature, and a prediction target.
- [ ] I can explain why the test set must stay separate during fitting.
- [ ] I can use a numerical `Pipeline` and categorical `Pipeline` inside `ColumnTransformer`.
- [ ] I can reuse a fitted transformer to prepare new, unseen observations.

**Lab 4 preview:** Combine this `preprocessor` with a regression estimator (`LinearRegression`, `DecisionTreeRegressor`, etc.) inside a *full modeling pipeline*. Compare predictions against a simple baseline and evaluate the held-out test set **only after** making modeling decisions using training data.

> **Practical note:** In Lab 4 you should recreate the split and preprocessing steps in your own notebook. Notebook variables do not carry over automatically between separate Jupyter notebooks.